# Predicting Late Responses to Consumer Financial Complaints
**Author:** Kejie Wang — **Platform:** Databricks / PySpark ML

Research question: Can complaint information available at submission predict a late company response?
Target: `Timely response?`: No = 1 (late), Yes = 0 (timely).

Source: https://www.consumerfinance.gov/data-research/consumer-complaints/
Download: https://files.consumerfinance.gov/ccdb/complaints.csv.zip
Uploaded snapshot: `complaints.csv`, **5,626,886,348 bytes**, 5.63 GB / 5.24 GiB uncompressed.
The ZIP itself is 356,704,969 bytes. A fresh download may differ; retain your snapshot.

**Before starting:** obtain the instructor's topic approval, as required by the rubric.
No model results are prefilled. All tables below are computed when the notebook runs.


## 1. Set up Databricks
1. Extract `complaints.csv.zip` on your computer.
2. In Databricks, upload **complaints.csv** to a Unity Catalog Volume that you can read and write.
3. Import this `.ipynb` into Workspace and attach compute that supports Spark ML training.
4. Edit the three `/Volumes/...` paths below, then run the cells in order.
5. Fill in actual compute details. Do not report the example placeholders as hardware specifications.

The default uses the **entire dataset**. Eight tuning experiments (four algorithms × two tuning methods)
can take substantial time. Use `QUICK_RUN=True` only to check the workflow, then set it back to False
and rerun all cells for the final experiment. Record the actual row counts and sampling setting.


In [ ]:
# EDIT THESE VALUES. Use an existing writable Unity Catalog Volume.
DATA_PATH = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/YOUR_VOLUME/complaints.csv"
OUTPUT_BASE = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/YOUR_VOLUME/complaints_project"
SPARKML_TEMP_DFS_PATH = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/YOUR_VOLUME/sparkml_temp"
COMPUTE_DETAILS = {
    "compute_type": "FILL IN: classic or serverless",
    "runtime": "FILL IN: Databricks runtime",
    "driver_type": "FILL IN or unavailable",
    "worker_type": "FILL IN or unavailable",
    "worker_count": "FILL IN or managed by serverless",
    "total_worker_cores": "FILL IN or unavailable",
    "worker_memory": "FILL IN or unavailable",
}
QUICK_RUN = False
QUICK_FRACTION = 0.02
SEED = 42
NUM_FOLDS = 3
HASH_DIMENSIONS = 512
PFI_MAX_ROWS = 10000
PFI_REPEATS = 3

import os
assert "YOUR_" not in DATA_PATH + OUTPUT_BASE + SPARKML_TEMP_DFS_PATH, "Edit the Volume paths first."
# Set before Spark ML imports/tuning, including on shared/serverless compute.
os.environ["SPARKML_TEMP_DFS_PATH"] = SPARKML_TEMP_DFS_PATH
os.makedirs(SPARKML_TEMP_DFS_PATH, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)


In [ ]:
import json, time, platform, uuid, random, statistics
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import FeatureHasher
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, TrainValidationSplit, ParamGridBuilder

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = Path(OUTPUT_BASE) / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
metadata = {
    "run_id": RUN_ID, "spark_version": spark.version,
    "python_version": platform.python_version(),
    "driver_visible_cpu_count_not_cluster_total": os.cpu_count(),
    "hardware": COMPUTE_DETAILS, "data_path": DATA_PATH,
    "actual_csv_bytes": os.path.getsize(DATA_PATH),
    "quick_run": QUICK_RUN, "quick_fraction": QUICK_FRACTION if QUICK_RUN else 1.0,
    "seed": SEED, "folds": NUM_FOLDS, "hash_dimensions": HASH_DIMENSIONS,
    "source_url": "https://www.consumerfinance.gov/data-research/consumer-complaints/",
}
print(json.dumps(metadata, indent=2))


## 2. Load and audit the data
Use an explicit string schema and fail on malformed records. The CSV must be extracted first;
Spark's CSV reader is not pointed at a ZIP archive. Missing or invalid target values are excluded,
never converted to timely responses. Complaint IDs must be present and unique.


In [ ]:
columns = ["Date received", "Product", "Sub-product", "Issue", "Sub-issue",
           "Company public response", "Company", "State", "ZIP code", "Tags",
           "Submitted via", "Date sent to company", "Company response to consumer",
           "Timely response?", "Complaint ID"]
schema = T.StructType([T.StructField(c, T.StringType(), True) for c in columns])
start = time.perf_counter()
raw = (spark.read.schema(schema).option("header", True).option("enforceSchema", False)
       .option("mode", "FAILFAST").option("multiLine", True).csv(DATA_PATH))
# multiLine safely handles quoted newlines but may limit parallel CSV ingestion.
raw_count = raw.count()
print("Raw rows:", raw_count, "Load/count seconds:", time.perf_counter() - start)
raw.groupBy("Timely response?").count().orderBy(F.desc("count")).show(truncate=False)
id_audit = raw.agg(F.countDistinct("Complaint ID").alias("unique_ids"),
                  F.sum(F.when(F.col("Complaint ID").isNull() |
                               (F.trim(F.col("Complaint ID")) == ""), 1).otherwise(0)).alias("missing_ids")).first()
assert id_audit.missing_ids == 0 and id_audit.unique_ids == raw_count, "Investigate duplicate/missing IDs first."


## 3. Submission-time features and reproducible splits
Exclude response outcomes, dispatch date, ZIP code, tags, and complaint ID from model inputs.
Company identifies the complaint recipient; product/issue and submission channel describe the filing.
The ID is used only for reproducible splitting. Parse dates with a tolerant cast.

Hash original categorical columns into 512 dimensions. Hashing needs no fitted vocabulary and handles
unseen categories, but collisions may reduce accuracy. All four models use the same representation.
Interpret original columns with permutation importance rather than treating hash bins as named features.

Use approximately 70% for training/tuning, 15% for model selection, and 15% for final testing.
This random holdout estimates performance on similar complaints; it does not establish future-year
performance. A future extension should use a chronological test period and time-aware tuning.


In [ ]:
target = F.lower(F.trim(F.col("Timely response?")))
valid = raw.where(target.isin("yes", "no")).withColumn("label", (target == "no").cast("double"))
base = valid.select(
    F.col("Complaint ID").alias("complaint_id"), "label",
    F.col("Product").alias("product"), F.col("Sub-product").alias("sub_product"),
    F.col("Issue").alias("issue"), F.col("Sub-issue").alias("sub_issue"),
    F.col("Company").alias("company"), F.col("State").alias("state"),
    F.col("Submitted via").alias("submitted_via"),
    F.expr("try_cast(`Date received` AS DATE)").alias("received_date")
)
for name, expr in [("received_year", F.year("received_date")),
                   ("received_month", F.month("received_date")),
                   ("received_weekday", F.dayofweek("received_date"))]:
    base = base.withColumn(name, expr.cast("string"))
FEATURES = ["product", "sub_product", "issue", "sub_issue", "company", "state",
            "submitted_via", "received_year", "received_month", "received_weekday"]
for c in FEATURES:
    base = base.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""),
                                    F.lit("__MISSING__")).otherwise(F.col(c)))
base = base.select("complaint_id", "label", *FEATURES)
if QUICK_RUN:
    base = base.sample(False, QUICK_FRACTION, seed=SEED)
base = base.withColumn("split_bucket", F.pmod(F.xxhash64("complaint_id", F.lit(SEED)), F.lit(10000)))
train = base.where("split_bucket < 7000").drop("split_bucket")
selection = base.where("split_bucket >= 7000 AND split_bucket < 8500").drop("split_bucket")
test = base.where("split_bucket >= 8500").drop("split_bucket")
split_rows = []
for name, frame in [("train", train), ("selection", selection), ("test", test)]:
    counts = {int(r.label): r['count'] for r in frame.groupBy("label").count().collect()}
    assert counts.get(0, 0) >= 20 and counts.get(1, 0) >= 20, "Too few examples of either class; increase data."
    split_rows.append({"split": name, "rows": sum(counts.values()),
                       "timely": counts[0], "late": counts[1],
                       "late_rate": counts[1] / sum(counts.values())})
split_table = pd.DataFrame(split_rows)
display(split_table)
metadata.update({"raw_rows": raw_count, "splits": split_rows})
split_table.to_csv(RUN_DIR / "split_counts.csv", index=False)


## 4. Metrics and baselines
Late response is the positive class. Tune by area under the precision-recall curve (PR-AUC).
Report **positive-class** precision, recall and F1, together with accuracy, ROC-AUC and PR-AUC.
Precision/recall use the models' default decision thresholds; no threshold is adjusted on the test set.
Class weights and resampling are not used in this baseline experiment.

A model predicting every complaint as timely has zero late-response recall. A constant ranking's
PR-AUC equals prevalence and ROC-AUC is 0.5 when both classes exist. Spark computes curve areas;
PR-AUC here should not be described as average precision from a different library.


In [ ]:
pr_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                             metricName="areaUnderPR")
roc_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                              metricName="areaUnderROC")
def evaluate_predictions(pred):
    r = pred.agg(
        F.sum(F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1).otherwise(0)).alias("tp"),
        F.sum(F.when((F.col("label") == 0) & (F.col("prediction") == 1), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("label") == 1) & (F.col("prediction") == 0), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("label") == 0) & (F.col("prediction") == 0), 1).otherwise(0)).alias("tn")
    ).first().asDict()
    tp, fp, fn, tn = [r[k] for k in ["tp", "fp", "fn", "tn"]]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return dict(r, accuracy=(tp+tn)/(tp+fp+fn+tn), late_precision=precision,
                late_recall=recall, late_f1=2*precision*recall/(precision+recall) if precision+recall else 0.0,
                roc_auc=roc_evaluator.evaluate(pred), pr_auc=pr_evaluator.evaluate(pred))
baseline = [{"split": r["split"], "all_timely_accuracy": 1-r["late_rate"],
             "all_timely_late_recall": 0.0, "constant_score_pr_auc": r["late_rate"],
             "constant_score_roc_auc": 0.5} for r in split_rows]
display(pd.DataFrame(baseline))


## 5. Four algorithms, each tuned with both required methods
Run three-fold CrossValidator **and** TrainValidationSplit for every algorithm (eight experiments).
Both methods receive only the training split and use the same candidate grids.
Each selected configuration is refitted on the full training split by Spark's tuner.

`fit_seconds_including_refit` includes the entire tuning process and its automatic refit.
The separately timed refit measures one training run with the chosen parameters; it is **not added**
to tuning time. Selection scoring time includes input reads, feature transformation and scoring.
No explicit DataFrame caching is used, for compatibility with managed compute. Hardware, shared load,
and file reads affect all timings. These are end-to-end measurements, not isolated CPU benchmarks.


In [ ]:
def model_specs():
    lr = LogisticRegression(maxIter=60, standardization=True)
    dt = DecisionTreeClassifier(seed=SEED, maxBins=32)
    rf = RandomForestClassifier(seed=SEED, numTrees=40, maxBins=32)
    gbt = GBTClassifier(seed=SEED, maxIter=25, maxBins=32)
    return [
        ("LogisticRegression", lr, ParamGridBuilder().addGrid(lr.regParam, [0.001, 0.01]).build()),
        ("DecisionTree", dt, ParamGridBuilder().addGrid(dt.maxDepth, [4, 7]).build()),
        ("RandomForest", rf, ParamGridBuilder().addGrid(rf.maxDepth, [4, 7]).build()),
        ("GBT", gbt, ParamGridBuilder().addGrid(gbt.maxDepth, [3, 5]).build()),
    ]

experiments, fitted_models = [], {}
for model_name, estimator, grid in model_specs():
    for method in ["CrossValidator", "TrainValidationSplit"]:
        experiment_id = model_name + "_" + method
        hasher = FeatureHasher(inputCols=FEATURES, outputCol="features", numFeatures=HASH_DIMENSIONS)
        pipeline = Pipeline(stages=[hasher, estimator])
        common = dict(estimator=pipeline, estimatorParamMaps=grid,
                      evaluator=pr_evaluator, parallelism=1, seed=SEED, collectSubModels=False)
        if method == "CrossValidator":
            tuner = CrossValidator(numFolds=NUM_FOLDS, **common)
        else:
            tuner = TrainValidationSplit(trainRatio=0.8, **common)
        print("Starting:", experiment_id, flush=True)
        started = time.perf_counter()
        tuned = tuner.fit(train)
        fit_seconds = time.perf_counter() - started
        internal_scores = list(tuned.avgMetrics if method == "CrossValidator" else tuned.validationMetrics)
        best_index = max(range(len(internal_scores)), key=lambda i: internal_scores[i])
        best_params = grid[best_index]
        params_text = json.dumps({p.name: v for p, v in best_params.items()}, sort_keys=True)
        started = time.perf_counter()
        chosen = pipeline.copy(best_params).fit(train)
        refit_seconds = time.perf_counter() - started
        started = time.perf_counter()
        # Aggregating predictions forces model scoring; count() alone can prune prediction work.
        predictions = chosen.transform(selection)
        predictions.agg(F.sum("prediction").alias("predicted_positive_count")).collect()
        scoring_seconds = time.perf_counter() - started
        metrics = evaluate_predictions(predictions)
        row = dict(experiment_id=experiment_id, model=model_name, tuning_method=method,
                   params=params_text, fit_seconds_including_refit=fit_seconds,
                   single_refit_seconds=refit_seconds, selection_scoring_seconds=scoring_seconds,
                   internal_validation_pr_auc=float(internal_scores[best_index]),
                   **{"selection_"+k: v for k,v in metrics.items()})
        experiments.append(row)
        fitted_models[experiment_id] = chosen
        # Save after every experiment so completed work remains available if a later fit fails.
        pd.DataFrame(experiments).to_csv(RUN_DIR / "selection_comparison.csv", index=False)
        print(experiment_id, "selection PR-AUC:", metrics["pr_auc"], flush=True)
comparison = pd.DataFrame(experiments).sort_values("selection_pr_auc", ascending=False)
display(comparison)


## 6. Lock the winner before final testing
Choose the experiment with highest **selection-set** PR-AUC, using selection F1 and experiment name
only to break ties. Do not change this choice after inspecting test results.
Report every experiment's test results transparently; the best algorithm in the project is the
preselected winner, even if a different model happens to score higher on the test set.


In [ ]:
ranked = sorted(experiments, key=lambda r: (-r["selection_pr_auc"], -r["selection_late_f1"], r["experiment_id"]))
WINNER_ID = ranked[0]["experiment_id"]
best_model = fitted_models[WINNER_ID]
metadata["selected_experiment"] = WINNER_ID
metadata["selection_rule"] = "selection PR-AUC descending; selection late F1 descending; experiment ID ascending"
print("LOCKED winner:", WINNER_ID)
with open(RUN_DIR / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

test_rows = []
for row in experiments:
    chosen = fitted_models[row["experiment_id"]]
    started = time.perf_counter()
    pred = chosen.transform(test)
    pred.agg(F.sum("prediction")).collect()
    score_seconds = time.perf_counter() - started
    metrics = evaluate_predictions(pred)
    test_rows.append(dict(experiment_id=row["experiment_id"], model=row["model"],
                          tuning_method=row["tuning_method"],
                          selected_before_test=row["experiment_id"] == WINNER_ID,
                          fit_seconds_including_refit=row["fit_seconds_including_refit"],
                          single_refit_seconds=row["single_refit_seconds"],
                          test_scoring_seconds=score_seconds, **metrics))
test_table = pd.DataFrame(test_rows)
display(test_table)
test_table.to_csv(RUN_DIR / "test_comparison.csv", index=False)
print("Selected model's final test result:")
display(test_table[test_table.selected_before_test])


## 7. Permutation feature importance of original columns
For the locked winner, compute the drop in PR-AUC after shuffling one original feature group.
Use a deterministic bounded test sample (at most 10,000 rows) and repeat each shuffle three times.
Only these bounded raw rows go to the driver; training and model scoring remain in Spark ML.
Shuffle the date components together so year/month/weekday remain paired.

Positive importance means shuffling hurt performance; negative values may reflect noise.
Correlated product/issue features can substitute for each other, reducing individual importance.
Shuffling can produce unusual feature combinations. Importance measures predictive reliance,
not a causal effect, and must not be used to retune the model after seeing the test results.


In [ ]:
sample_rows = (test.orderBy(F.xxhash64("complaint_id", F.lit(SEED + 99)))
               .limit(PFI_MAX_ROWS).collect())
assert len({r.label for r in sample_rows}) == 2, "Increase PFI_MAX_ROWS to include both classes."
pfi_schema = test.schema
sample_dicts = [r.asDict() for r in sample_rows]
sample_df = spark.createDataFrame(sample_rows, schema=pfi_schema)
baseline_pr = pr_evaluator.evaluate(best_model.transform(sample_df))
groups = {c: [c] for c in FEATURES if not c.startswith("received_")}
groups["Date received"] = ["received_year", "received_month", "received_weekday"]
importance_rows = []
for group_index, (group, cols) in enumerate(groups.items()):
    drops = []
    for repeat in range(PFI_REPEATS):
        order = list(range(len(sample_dicts)))
        random.Random(SEED + group_index * 100 + repeat).shuffle(order)
        shuffled = [dict(r) for r in sample_dicts]
        for i, j in enumerate(order):
            for col in cols:
                shuffled[i][col] = sample_dicts[j][col]
        shuffled_df = spark.createDataFrame(shuffled, schema=pfi_schema)
        shuffled_pr = pr_evaluator.evaluate(best_model.transform(shuffled_df))
        drops.append(baseline_pr - shuffled_pr)
    importance_rows.append(dict(feature=group, mean_pr_auc_drop=statistics.mean(drops),
                                std_pr_auc_drop=statistics.pstdev(drops),
                                repeats=PFI_REPEATS, sample_rows=len(sample_rows),
                                baseline_sample_pr_auc=baseline_pr))
importance = pd.DataFrame(importance_rows).sort_values("mean_pr_auc_drop", ascending=False)
display(importance)
importance.to_csv(RUN_DIR / "permutation_importance.csv", index=False)


## 8. Error analysis and measured findings
Compare the winner's errors across product and submission channel. Show group counts to avoid
interpreting a small subgroup as conclusive. These are descriptive test-set results.


In [ ]:
best_predictions = best_model.transform(test)
for group in ["product", "submitted_via"]:
    breakdown = (best_predictions.groupBy(group).agg(
        F.count("*").alias("rows"),
        F.sum("label").alias("actual_late"),
        F.sum(F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1).otherwise(0)).alias("caught_late"),
        F.sum(F.when((F.col("label") == 0) & (F.col("prediction") == 1), 1).otherwise(0)).alias("false_alerts"))
        .withColumn("late_recall", F.when(F.col("actual_late") > 0, F.col("caught_late") / F.col("actual_late")))
        .orderBy(F.desc("rows")))
    display(breakdown)
    breakdown.toPandas().to_csv(RUN_DIR / (group + "_error_analysis.csv"), index=False)

winner_metrics = next(r for r in test_rows if r["selected_before_test"])
top_features = ", ".join(importance.head(3).feature.tolist())
findings = f"""# Measured project findings
Selected experiment: {WINNER_ID}
Selection criterion: held-out selection PR-AUC.
Final test PR-AUC: {winner_metrics['pr_auc']:.6f}
Final test ROC-AUC: {winner_metrics['roc_auc']:.6f}
Late-response precision: {winner_metrics['late_precision']:.6f}
Late-response recall: {winner_metrics['late_recall']:.6f}
Late-response F1: {winner_metrics['late_f1']:.6f}
Accuracy: {winner_metrics['accuracy']:.6f}
Most influential feature groups on the permutation sample: {top_features}.
Quick run: {QUICK_RUN}. Use a full-data run for final reported results.
Interpretation: predictive associations in published complaints, not causal effects or market-wide rates.
"""
(RUN_DIR / "measured_findings.md").write_text(findings)
print(findings)
print("Outputs:", str(RUN_DIR))


## 9. Optional: save the winning Spark pipeline
Enable only after the experiment succeeds. A unique run directory prevents overwriting previous runs.
Some managed environments restrict model persistence; the CSV tables remain the reporting outputs.


In [ ]:
SAVE_MODEL = False
if SAVE_MODEL:
    best_model.write().save(str(RUN_DIR / "best_spark_pipeline"))
    print("Saved winner pipeline.")


## 10. GitHub, slides and term paper checklist
Suggested repository contents:
- `README.md`: research question, data source, setup, full-data size, actual compute, measured results, limitations.
- This executed `.ipynb` notebook, including output tables.
- `results/`: generated comparison tables, split counts, metadata, importance and findings.
- Presentation slides with **all text at least 28 pt**, including tables and figures.
- Term paper with introduction, related work, dataset, methods, experiments, results, limitations and conclusion.
- `data/README.md`: official download URL, snapshot date, CSV size, extraction and upload instructions.
  Ask the instructor whether an externally hosted full dataset plus a small sample meets the dataset deliverable.

Slide sequence: title/research question; originality and use case; source and dataset size; class balance;
submission-time features; hardware specifications; four models and tuning; comparison results;
locked winner and confusion matrix; permutation importance; error analysis/limitations; conclusion and GitHub/data URLs.
Split wide result tables across slides to preserve 28-point text. Never invent hardware or accuracy.

Original contribution to emphasize: a reproducible late-response screening experiment with both tuning
methods, explicit prevention of outcome leakage, original-column importance and error analysis by filing category.
This is an application contribution, not a claim to have invented the algorithms.

Limitations: complaints are a selected population; labels may reflect operational practices; random holdouts
do not establish temporal generalization; hashing loses some detail; rare late responses can limit recall;
feature importance does not establish causality. Model outputs are an academic screening experiment.

References:
- CFPB data: https://www.consumerfinance.gov/data-research/consumer-complaints/
- Spark model tuning: https://spark.apache.org/docs/latest/ml-tuning.html
- Spark FeatureHasher: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.FeatureHasher.html
- Spark classification: https://spark.apache.org/docs/latest/ml-classification-regression.html

Troubleshooting:
- `SPARKML_TEMP_DFS_PATH`: the setup cell sets a writable `/Volumes/...` location before ML imports.
  If your session previously imported Spark ML before setting it, restart Python and rerun in order.
- Permission errors: use a Volume and compute mode your Databricks account permits.
- Missing names: run all cells from the beginning.
- Long training: use quick mode to verify setup; retain full-data settings for the final experiment.
- CSV ingestion is slow: multiline CSV parsing can restrict parallelism. After validating your snapshot,
  convert it once to Delta in your Databricks environment and adapt the loader for subsequent runs.
